# 02E — Exon architecture & hotspots


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📊 exon distribution


In [3]:
exon_counts = d['exon_num'].dropna().astype(int).value_counts().sort_index().reset_index()
exon_counts.columns = ['exon', 'count']
fig = px.bar(exon_counts, x='exon', y='count', title='Exon distribution')
fig.show()


## 2. 📊 exon pathogenic stacked 🛡


In [4]:
tmp = d.dropna(subset=['exon_num']).copy()
tmp['exon'] = tmp['exon_num'].astype(int)
fig = px.histogram(tmp, x='exon', color='pathogenic', barmode='stack', title='Exon by pathogenic status')
fig.show()


## 3. 📊 exon pathogenic fraction


In [5]:
tmp = d.dropna(subset=['exon_num']).copy()
tmp['exon'] = tmp['exon_num'].astype(int)
tbl = tmp.groupby('exon').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('var_id', 'size')).reset_index()
fig = px.line(tbl, x='exon', y='pathogenic_fraction', markers=True, title='Pathogenic fraction by exon')
fig.show()


## 4. 📊 lollipop exon counts


In [11]:
tbl = d['exon_num'].dropna().astype(int).value_counts().sort_index().reset_index()
tbl.columns = ['exon', 'count']

fig = go.Figure()
for exon_value, count_value in zip(tbl['exon'], tbl['count']):
    fig.add_shape(
        type='line',
        x0=exon_value,
        x1=exon_value,
        y0=0,
        y1=count_value,
        line=dict(color='#4C78A8', width=1.5),
    )

fig.add_trace(
    go.Scatter(
        x=tbl['exon'],
        y=tbl['count'],
        mode='markers',
        marker=dict(size=8, color='#E45756'),
        name='Variant count',
        hovertemplate='Exon %{x}<br>Count %{y}<extra></extra>',
    )
)

fig.update_layout(
    title='Lollipop graph: exon counts',
    xaxis_title='Exon',
    yaxis_title='Count',
)
fig.update_yaxes(rangemode='tozero')
fig.show()


## 5. 📊 exon vs phenotype heatmap


In [7]:
tmp = d.dropna(subset=['exon_num']).copy()
tmp['exon'] = tmp['exon_num'].astype(int)
heat = pd.crosstab(tmp['exon'], tmp['phenotype'])
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Blues', title='Exon vs phenotype')
fig.show()


## 6. 🧪 χ² exon ~ phenotype


In [12]:
tmp = d.dropna(subset=['exon_num']).copy()
tmp['exon'] = tmp['exon_num'].astype(int)
tab, chi2, p, dof = chi2_table(tmp, 'exon', 'phenotype')
display(tab.head(30))
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


phenotype,BMD,DMD
exon,,
1,15,41
2,13,70
3,12,84
4,16,98
5,9,70
6,16,138
7,15,107
8,22,147
9,25,130


,value
chi2,2.183096e+02
dof,7.800000e+01
p_value,3.195194e-15


## 7. 📊 cumulative exon curve


In [13]:
tbl = d['exon_num'].dropna().astype(int).value_counts().sort_index().reset_index()
tbl.columns = ['exon', 'count']
tbl['cumulative'] = tbl['count'].cumsum() / tbl['count'].sum()
fig = px.line(tbl, x='exon', y='cumulative', markers=True, title='Cumulative exon curve')
fig.show()


## 8. 📊 exon vs pathogenic fraction scatter


In [14]:
tmp = d.dropna(subset=['exon_num']).copy()
tmp['exon'] = tmp['exon_num'].astype(int)
tbl = tmp.groupby('exon').agg(pathogenic_fraction=('pathogenic', 'mean'), count=('var_id', 'size')).reset_index()
fig = px.scatter(tbl, x='exon', y='pathogenic_fraction', size='count', title='Exon vs pathogenic fraction')
fig.show()


## 9. 🧪 Spearman exon vs pathogenic ⭐


In [15]:
tmp = d.dropna(subset=['exon_num']).copy()
tmp['exon'] = tmp['exon_num'].astype(int)
tbl = tmp.groupby('exon').agg(pathogenic_fraction=('pathogenic', 'mean')).reset_index()
tmp, rho, p = spearman_xy(tbl, 'exon', 'pathogenic_fraction')
display(pd.Series({'n_exons': len(tmp), 'spearman_rho': rho, 'p_value': p}).to_frame('value'))


,value
n_exons,79.000000
spearman_rho,-0.074636
p_value,0.513295


## 10. 📊 distal hotspot (45–55) enrichment 🛡 ⭐


In [16]:
tbl = d.groupby('is_hotspot_distal').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('var_id', 'size')).reset_index()
display(tbl)
fig = px.bar(tbl, x='is_hotspot_distal', y='pathogenic_fraction', hover_data=['n'], title='Distal hotspot pathogenic enrichment (45-55)')
fig.show()


,is_hotspot_distal,pathogenic_fraction,n
0,False,0.282684,9523
1,True,0.406723,1785


## 11. 🧪 Fisher distal hotspot ⭐


In [17]:
tab, odds, p, ci = fisher_bool(d, 'is_hotspot_distal', 'pathogenic')
display(tab)
display(pd.Series({'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


pathogenic,False,True
is_hotspot_distal,,
False,6831,2692
True,1059,726


,value
odds_ratio,1.739602e+00
p_value,1.334826e-24
or_ci_low,1.567078e+00
or_ci_high,1.931119e+00


## 12. 📊 proximal hotspot (3–9) enrichment ⭐


In [18]:
tbl = d.groupby('is_hotspot_proximal').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('var_id', 'size')).reset_index()
display(tbl)
fig = px.bar(tbl, x='is_hotspot_proximal', y='pathogenic_fraction', hover_data=['n'], title='Proximal hotspot pathogenic enrichment (3-9)')
fig.show()


,is_hotspot_proximal,pathogenic_fraction,n
0,False,0.303561,10222
1,True,0.290055,1086


## 13. 🧪 Fisher proximal hotspot ⭐


In [19]:
tab, odds, p, ci = fisher_bool(d, 'is_hotspot_proximal', 'pathogenic')
display(tab)
display(pd.Series({'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


pathogenic,False,True
is_hotspot_proximal,,
False,7119,3103
True,771,315


,value
odds_ratio,0.937332
p_value,0.366407
or_ci_low,0.816770
or_ci_high,1.075690


## 14. 📋 OR + CI hotspots ⭐


In [20]:
rows = []
for label, col in [('distal_45_55', 'is_hotspot_distal'), ('proximal_3_9', 'is_hotspot_proximal')]:
    tab, odds, p, ci = fisher_bool(d, col, 'pathogenic')
    rows.append({'hotspot': label, 'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]})
display(pd.DataFrame(rows))


,hotspot,odds_ratio,p_value,or_ci_low,or_ci_high
0,distal_45_55,1.739602,1.334826e-24,1.567078,1.931119
1,proximal_3_9,0.937332,3.664073e-01,0.816770,1.075690


## 15. 📊 exon vs mutation_type


In [21]:
tmp = d.dropna(subset=['exon_num']).copy()
tmp['exon'] = tmp['exon_num'].astype(int)
heat = pd.crosstab(tmp['exon'], tmp['mutation_type'])
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Viridis', title='Exon vs mutation_type')
fig.show()


## 16. 🧪 χ² exon ~ mutation_type


In [22]:
tmp = d.dropna(subset=['exon_num']).copy()
tmp['exon'] = tmp['exon_num'].astype(int)
tab, chi2, p, dof = chi2_table(tmp, 'exon', 'mutation_type')
display(tab.head(30))
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


mutation_type,frameshift,large_deletion,missense,nonsense,splice
exon,,,,,
1,5,23,6,2,15
2,10,30,20,2,16
3,14,2,29,7,10
4,8,34,27,5,13
5,9,5,38,4,18
6,24,3,73,10,15
7,11,42,35,14,15
8,13,2,72,17,21
9,6,31,60,6,22


,value
chi2,1.784264e+03
dof,3.120000e+02
p_value,1.866338e-204


## 17. 📊 exon vs frame_status


In [23]:
tmp = d.dropna(subset=['exon_num']).copy()
tmp['exon'] = tmp['exon_num'].astype(int)
heat = pd.crosstab(tmp['exon'], tmp['frame'])
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Magma', title='Exon vs frame_status')
fig.show()


## 18. 🧪 χ² exon ~ frame


In [24]:
tmp = d.dropna(subset=['exon_num']).copy()
tmp['exon'] = tmp['exon_num'].astype(int)
tab, chi2, p, dof = chi2_table(tmp, 'exon', 'frame')
display(tab.head(30))
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


frame,in-frame,out-of-frame
exon,,
1,6,45
2,20,58
3,29,33
4,27,60
5,38,36
6,73,52
7,35,82
8,72,53
9,60,65


,value
chi2,4.541517e+02
dof,7.800000e+01
p_value,1.893532e-54


## 19. 📊 exon vs interval_length


In [25]:
tmp = d[['exon_num', 'interval_length_num']].dropna().copy()
tmp['exon'] = tmp['exon_num'].astype(int)
fig = px.scatter(tmp, x='exon', y='interval_length_num', opacity=0.5, title='Exon vs interval_length')
fig.update_yaxes(type='log')
fig.show()


## 20. 🧪 Spearman exon vs interval_length 📖


In [26]:
tmp = d[['exon_num', 'interval_length_num']].dropna().copy()
tmp['exon'] = tmp['exon_num'].astype(int)
tmp, rho, p = spearman_xy(tmp, 'exon', 'interval_length_num')
display(pd.Series({'n': len(tmp), 'spearman_rho': rho, 'p_value': p}).to_frame('value'))


,value
n,2328.000000
spearman_rho,0.072261
p_value,0.000484


## 21. 📊 exon ECDF


In [27]:
vals = d['exon_num'].dropna().astype(float).values
x, y = ecdf_xy(vals)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode='lines'))
fig.update_layout(title='ECDF of exon positions', xaxis_title='Exon', yaxis_title='ECDF')
fig.show()


## 22. 📊 exon density smoothing 📖


In [28]:
counts = d['exon_num'].dropna().astype(int).value_counts().reindex(range(1, 80), fill_value=0).rename_axis('exon').reset_index(name='count')
counts['smooth'] = counts['count'].rolling(5, center=True, min_periods=1).mean()
fig = px.line(counts, x='exon', y=['count', 'smooth'], title='Exon density and smoothed curve')
fig.show()
